# Pipeline LightGBM v2 — clusters k-means DTW + K modelos (Labo 3, 202002)

Respeta el esquema de la clase (pizarra):

  DATASET (zero-fill cátedra z601)  ->  ESCALADO + FEATURE ENGINEERING
    ->  columna CLASE (t+2, NA en nov/dic-2019) + columna CLUSTER
    ->  particiona por cluster en M_1..M_K
    ->  cada M_i: OPTUNA + ENTRENAMIENTO FINAL (walk-forward, train hasta 201910)
    ->  PREDICT del modelo correspondiente a cada serie
  max_bin=1023, distribución Tweedie.

Diferencias con el v1:
 - **FE recortado a 15 features** (menos cómputo; sin canaritos).
 - **Clustering = k-means con DTW + centroides DBA** (motor de Rosario, mejorado):
   k se ELIGE por silhouette + balance sobre `lista_k` (NADA de k=7 hardcodeado),
   y TODAS las series se asignan a un cluster (las cortas también) en paralelo.
 - **Optuna tunea hiperparámetros POR cluster**; K modelos separados.
 - **Magic multiplier**: al final se emiten 3 submissions (×0.9, ×1.0, ×1.1)
   para subir las tres a Kaggle.

Leyes que se respetan: sólo la CLASE mira el futuro (t+2); features y escala
usan sólo <= t; escalado relativo `tn / media_expandida(<=t)`, 0/0:=0; clase
escalada `tn(t+2)/s(t)`; WAPE evaluado a nivel PRODUCTO sobre los 780;
validación walk-forward por "febreros". Notebook resumible (checkpoints atómicos).

## 0) Setup en la nube (correr una vez)

Instala dependencias. En **Linux/GCP** el wheel de `dtaidistance` trae la lib **C** (verificá `dtw_C=True` abajo; sin ella el k-means DTW es inusable a escala real). Con VM de 128 GB el clustering puede tardar horas: está OK.

**Ya viene en modo producción (`modo_test=False`)**: usa los ~1233 productos y los parámetros de escala reales. Si querés un smoke-test rápido en la VM antes del run completo, poné `modo_test=True` en la celda de Parámetros (usa pocos productos y termina en minutos), verificá que llega a los 3 `submission_x*.csv`, y volvé a `False`. Es resumible: cada etapa se saltea si su checkpoint ya existe (borrá la carpeta `pipe_v2/` del bucket para forzar un run limpio).

Definí `LABO3_BUCKET` si el bucket no está en `~/buckets/b1`. Para submitear automático las 3, poné `submit=True` (requiere `kaggle.json`); si no, subilas a mano desde `pipe_v2/`.

In [ ]:
%pip install -q optuna dtaidistance lightgbm polars duckdb pandas numpy scipy scikit-learn joblib
# import os; os.environ['LABO3_BUCKET'] = '/ruta/al/bucket'

In [ ]:
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

import duckdb
import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _HAY_OPTUNA = True
except Exception:
    _HAY_OPTUNA = False

from dtaidistance import dtw
from joblib import Parallel, delayed
from sklearn.metrics import silhouette_score

N_CORES = os.cpu_count() or 4

# ¿está la lib C de dtaidistance? (en Linux/GCP viene en el wheel; acelera ~100x)
try:
    dtw.distance_fast(np.zeros(4), np.zeros(4), window=2)
    _C_OK = True
except Exception:
    _C_OK = False

try:
    dtw.warping_path(np.zeros(5), np.zeros(5), window=2)
    _WP_WINDOW = True
except TypeError:
    _WP_WINDOW = False


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 Path("/content/buckets/b1"), Path("/home/ds/buckets/b1")):
        if Path(cand).is_dir():
            return Path(cand)
    local = Path.home() / "Desktop" / "labo3-2026ba"
    if local.is_dir():
        return local
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


def escribir_atomico(fn, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    fn(tmp)
    tmp.replace(path)


def leer_json(path: Path, intentos=5, espera=1.0):
    for i in range(intentos):
        try:
            return json.loads(path.read_text())
        except (OSError, json.JSONDecodeError):
            if i == intentos - 1:
                raise
            time.sleep(espera)

## 1) Parámetros

In [ ]:
PARAM = {
    "modo_test": False,              # LISTO PARA LA NUBE (run completo). Poner True = smoke test rapido en la VM.

    "horizonte": 2,
    "periodo_inferencia": 201912,    # parados aca -> predecir 202002
    "corte_train_final": 201910,     # ultimo periodo con clase (t+2 = 201912)
    "solo_productos_target": False,

    # ---- FE: 15 features (lista cerrada) ----
    "features": [
        "lag_1", "lag_2", "lag_3", "lag_12",
        "rmean_3", "rmean_12", "rstd_3",
        "nivel_relativo", "tendencia_3_12",
        "meses_consec_sin_compra", "ventas_ult12",
        "mes", "share_prod_en_cat3", "cat3", "brand",
    ],
    "cols_categoricas": ["cat3", "brand"],
    "eps": 1e-6,

    # ---- clustering k-means DTW + DBA ----
    "cl_escalado": "media",          # forma = tn/media (no tamaño)
    "cl_banda": 3,                   # Sakoe-Chiba (meses)
    "cl_corte": 201910,              # clustering causal: solo <= este periodo
    "cl_lista_k": [4, 5, 6, 7, 8],   # k candidatos (se elige el mejor)
    "cl_min_balance": 0.02,          # ningun cluster < 2% de las series
    "cl_muestra_k": 6000,            # series para ELEGIR k (silhouette)
    "cl_muestra_fit": 80000,         # series para AJUSTAR los centroides (None = todas)
    "cl_max_iter": 15, "cl_dba_iters": 2, "cl_tol": 0.01,

    # ---- LightGBM / Tweedie ----
    "max_bin": 1023,
    "num_threads": N_CORES,
    "objetivo_lgbm": "tweedie",
    "tweedie_vp_rango": (1.1, 1.6),

    # ---- Optuna (por cluster) ----
    "n_trials": 40,
    "semillas_ensemble": [102191, 314159],

    # ---- submit / magic multiplier ----
    "multiplicadores": [0.9, 1.0, 1.1],
    "clip_min": 0.0,
    "kaggle_competition": "labo-iii-2026-ba",
    "submit": True,   # sube las 3 automaticamente a Kaggle (requiere ~/.kaggle/kaggle.json)

    "semilla": 102191,
}

if PARAM["modo_test"]:
    PARAM.update({
        "n_productos_test": 15,
        "cl_lista_k": [2, 3],
        "cl_muestra_k": 250, "cl_muestra_fit": None,
        "n_trials": 4, "semillas_ensemble": [102191],
    })

BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / ("pipe_v2_test" if PARAM["modo_test"] else "pipe_v2")
DIR_RAW.mkdir(parents=True, exist_ok=True)
DIR_OUT.mkdir(parents=True, exist_ok=True)
print(f"BUCKET={BUCKET}\nOUT={DIR_OUT}\ncores={N_CORES} optuna={_HAY_OPTUNA} "
      f"dtw_C={_C_OK} wp_window={_WP_WINDOW}")

## 2) Datos crudos

In [ ]:
def descargar(archivo):
    dst = DIR_RAW / archivo
    if dst.exists():
        return
    url = f"https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{archivo}"
    subprocess.run(["wget", url, "-O", str(dst)], check=True)


for _a in ("sell-in.txt.gz", "tb_productos.txt", "product_id_apredecir201912.txt"):
    descargar(_a)

PROD_TARGET = set(
    pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt", separator="\t")["product_id"].to_list()
)
print("productos a predecir:", len(PROD_TARGET))

## 3) Preprocesamiento — densa zero-fill cátedra (z601)

Grid ⟨cliente, producto, periodo⟩ con `tn=0` donde no hubo venta, dentro de la
vida del producto (min..max de venta, forzado a 201912 para los 780) **y** a
partir del primer periodo del cliente. Meses contiguos -> `shift(k)` = lag exacto.

In [ ]:
PATH_PRE = DIR_OUT / "preprocesado.parquet"


def _construir_preprocesado():
    con = duckdb.connect(); con.execute("SET preserve_insertion_order=false")
    con.execute(f"""CREATE TABLE sellin AS
        SELECT CAST(customer_id AS INT) customer_id, CAST(product_id AS INT) product_id,
               CAST(periodo AS INT) periodo, CAST(tn AS DOUBLE) tn
        FROM read_csv_auto('{DIR_RAW/'sell-in.txt.gz'}')""")
    con.execute(f"""CREATE TABLE apredecir AS
        SELECT CAST(product_id AS INT) product_id FROM read_csv_auto('{DIR_RAW/'product_id_apredecir201912.txt'}')""")

    filtro = ""
    if PARAM["solo_productos_target"]:
        filtro = "WHERE product_id IN (SELECT product_id FROM apredecir)"
    elif PARAM["modo_test"]:
        filtro = (f"WHERE product_id IN (SELECT product_id FROM apredecir "
                  f"ORDER BY product_id LIMIT {PARAM['n_productos_test']})")

    con.execute(f"""CREATE TABLE base AS
        SELECT customer_id, product_id, periodo, SUM(tn) tn FROM sellin {filtro}
        GROUP BY customer_id, product_id, periodo""")
    con.execute("CREATE TABLE periodos AS SELECT DISTINCT periodo FROM base")
    con.execute("""CREATE TABLE vida_prod AS
        SELECT product_id, MIN(periodo) nace, MAX(periodo) muere FROM base GROUP BY product_id""")
    con.execute("""UPDATE vida_prod SET muere = 201912
        WHERE product_id IN (SELECT product_id FROM apredecir) AND muere < 201912""")
    con.execute("""CREATE TABLE primer_cli AS
        SELECT customer_id, MIN(periodo) nace_cli FROM base GROUP BY customer_id""")
    con.execute("""CREATE TABLE grid AS
        SELECT c.customer_id, v.product_id, p.periodo
        FROM vida_prod v JOIN periodos p ON p.periodo BETWEEN v.nace AND v.muere
        CROSS JOIN primer_cli c WHERE p.periodo >= c.nace_cli""")
    con.execute("""CREATE TABLE densa AS
        SELECT g.customer_id, g.product_id, g.periodo, COALESCE(b.tn, 0.0) tn
        FROM grid g LEFT JOIN base b USING (customer_id, product_id, periodo)""")

    prods = con.execute(f"""SELECT CAST(product_id AS INT) product_id, cat3, brand
        FROM read_csv_auto('{DIR_RAW/'tb_productos.txt'}')""").df()
    df = con.execute("SELECT * FROM densa ORDER BY customer_id, product_id, periodo").pl()
    con.close()

    df = df.join(pl.from_pandas(prods), on="product_id", how="left")
    MULT = 100_000
    return df.with_columns(
        (pl.col("customer_id").cast(pl.Int64) * MULT + pl.col("product_id").cast(pl.Int64)).alias("agrupa_id")
    )


if PATH_PRE.exists():
    print("[preprocesado] RESUME"); df_pre = pl.read_parquet(PATH_PRE)
else:
    df_pre = _construir_preprocesado()
    escribir_atomico(lambda t: df_pre.write_parquet(t), PATH_PRE)
print("preprocesado:", df_pre.shape, "| series:", df_pre["agrupa_id"].n_unique())

## 4) Escalado + Feature Engineering (15 features, causal)

`s(t)` = media expandida de tn hasta t (presente); target = `tn(t+2)/s(t)` (0/0:=0).
Las 15 features salen todas de <= t; `share_prod_en_cat3` usa sólo el periodo t.

In [ ]:
PATH_FE = DIR_OUT / "features.parquet"


def _fe(df: pl.DataFrame) -> pl.DataFrame:
    g = "agrupa_id"; eps = PARAM["eps"]
    df = df.sort([g, "periodo"])
    df = df.with_columns(
        pl.col("tn").cum_sum().over(g).alias("_cs"),
        (pl.int_range(pl.len()).over(g) + 1).alias("_n"),
        pl.int_range(pl.len()).over(g).alias("_rn"),
    ).with_columns((pl.col("_cs") / pl.col("_n")).alias("s"))

    tn1 = pl.col("tn").shift(1).over(g)
    df = df.with_columns([
        pl.col("tn").shift(1).over(g).alias("lag_1"),
        pl.col("tn").shift(2).over(g).alias("lag_2"),
        pl.col("tn").shift(3).over(g).alias("lag_3"),
        pl.col("tn").shift(12).over(g).alias("lag_12"),
        tn1.rolling_mean(3, min_samples=1).alias("rmean_3"),
        tn1.rolling_mean(12, min_samples=1).alias("rmean_12"),
        tn1.rolling_std(3, min_samples=2).alias("rstd_3"),
        (pl.col("tn") > 0).cast(pl.Int32).rolling_sum(12, min_samples=1).over(g).shift(1).over(g).alias("ventas_ult12"),
        (pl.col("periodo") % 100).alias("mes"),
    ])
    df = df.with_columns([
        (pl.col("lag_1") / (pl.col("s") + eps)).alias("nivel_relativo"),
        (pl.col("rmean_3") / (pl.col("rmean_12") + eps)).alias("tendencia_3_12"),
        pl.when(pl.col("tn") > 0).then(pl.col("_rn")).otherwise(None).forward_fill().over(g).alias("_uv"),
    ])
    df = df.with_columns(
        (pl.col("_rn") - pl.col("_uv").fill_null(-1)).alias("meses_consec_sin_compra")
    )
    df = df.with_columns([
        pl.col("tn").sum().over(["product_id", "periodo"]).alias("_tn_prod"),
        pl.col("tn").sum().over(["cat3", "periodo"]).alias("_tn_cat3"),
    ]).with_columns(
        (pl.col("_tn_prod") / (pl.col("_tn_cat3") + eps)).alias("share_prod_en_cat3")
    )
    # clase (unico futuro) y su escalado
    df = df.with_columns(pl.col("tn").shift(-PARAM["horizonte"]).over(g).alias("clase_raw"))
    df = df.with_columns(
        pl.when(pl.col("s") > eps).then(pl.col("clase_raw") / pl.col("s")).otherwise(0.0).alias("clase_scaled")
    )
    keep = (["agrupa_id", "customer_id", "product_id", "periodo", "tn", "s",
             "clase_raw", "clase_scaled"] + PARAM["features"])
    return df.select([c for c in keep if c in df.columns])


if PATH_FE.exists():
    print("[features] RESUME"); df_fe = pl.read_parquet(PATH_FE)
else:
    df_fe = _fe(df_pre)
    escribir_atomico(lambda t: df_fe.write_parquet(t), PATH_FE)
print("features:", df_fe.shape, "|", len(PARAM["features"]), "features")

## 5) Clustering k-means DTW + DBA  (motor de Rosario, mejorado)

Serie por par = `tn/media` desde su primera venta, sólo `<= cl_corte` (causal).
Mejoras: `use_pruning`, asignación paralela a todos los núcleos, elección de k
por silhouette+balance sobre una muestra, y **todas** las series reciben cluster
(los centroides finales se ajustan sobre `cl_muestra_fit` y se asigna el resto).

In [ ]:
PATH_CL = DIR_OUT / "clusters.parquet"
BANDA = PARAM["cl_banda"]


def _banda(a, b):
    return max(int(BANDA), abs(len(a) - len(b)))


def _d(a, b):
    if _C_OK:
        d = dtw.distance_fast(a, b, window=_banda(a, b))
    else:
        d = dtw.distance(a, b, window=_banda(a, b), use_c=False)
    return d if np.isfinite(d) else np.inf


def _sanear(D):
    """reemplaza distancias no finitas (banda infactible) por un valor grande finito."""
    mal = ~np.isfinite(D)
    if mal.any():
        fin = D[~mal]
        D[mal] = (fin.max() * 10.0) if fin.size else 1.0
    return D


def _dvec(s, centros):
    """distancias de s a cada centro, saneadas (sin inf/nan)."""
    return _sanear(np.array([_d(s, c) for c in centros], dtype=np.float64))


def _serie(x: np.ndarray):
    """tn/media desde la primera venta; None si nunca vendió en <= corte."""
    nz = np.nonzero(x > 0)[0]
    if len(nz) == 0:
        return None
    x = np.ascontiguousarray(x[nz[0]:], dtype=np.float64)
    m = x.mean()
    return x / m if m > PARAM["eps"] else x


def _asignar(series, centros):
    def uno(s):
        return int(np.argmin(_dvec(s, centros)))
    return np.array(Parallel(n_jobs=N_CORES, prefer="threads", batch_size=256)(
        delayed(uno)(s) for s in series))


def _dba(miembros, centro, iters):
    centro = np.ascontiguousarray(centro, dtype=np.float64)
    if not miembros:
        return centro
    T = len(centro)
    for _ in range(iters):
        acum = np.zeros(T); cnt = np.zeros(T)
        for s in miembros:
            w = _banda(centro, s)
            path = dtw.warping_path(centro, s, window=w) if _WP_WINDOW else dtw.warping_path(centro, s)
            for i, j in path:
                acum[i] += s[j]; cnt[i] += 1.0
        centro = np.where(cnt > 0, acum / np.maximum(cnt, 1.0), centro)
        centro = np.ascontiguousarray(centro, dtype=np.float64)
    return centro


def _kmeanspp(series, k, rng):
    centros = [series[int(rng.integers(len(series)))].copy()]
    dmin = _sanear(np.array([_d(s, centros[0]) for s in series], dtype=np.float64))
    for _ in range(1, k):
        p = dmin ** 2; tot = p.sum()
        idx = int(rng.integers(len(series))) if not np.isfinite(tot) or tot <= 0 \
            else int(rng.choice(len(series), p=p / tot))
        centros.append(series[idx].copy())
        dmin = np.minimum(dmin, _sanear(np.array([_d(s, centros[-1]) for s in series], dtype=np.float64)))
    return centros


def _kmeans_dtw(series, k, semilla):
    rng = np.random.default_rng(semilla)
    centros = _kmeanspp(series, k, rng)
    lab = np.full(len(series), -1)
    for it in range(PARAM["cl_max_iter"]):
        lab_new = _asignar(series, centros)
        cambios = int((lab_new != lab).sum()); lab = lab_new
        for j in range(k):  # cluster vacio -> re-seed con la serie mas lejana
            if not np.any(lab == j):
                lab[int(rng.integers(len(series)))] = j
        centros = [_dba([series[i] for i in np.flatnonzero(lab == j)], centros[j], PARAM["cl_dba_iters"])
                   for j in range(k)]
        if cambios / len(series) < PARAM["cl_tol"]:
            break
    return lab, centros


def _matriz(series):
    n = len(series); D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            D[i, j] = D[j, i] = _d(series[i], series[j])
    D = _sanear(D)
    np.fill_diagonal(D, 0.0)
    return D


def _construir_clusters(df: pl.DataFrame) -> pl.DataFrame:
    hist = df.filter(pl.col("periodo") <= PARAM["cl_corte"])
    piv = hist.group_by("agrupa_id").agg(
        pl.col("tn").sort_by("periodo").alias("serie"),
        pl.col("tn").sum().alias("tn_total"))
    ids = piv["agrupa_id"].to_list()
    series_raw = [np.asarray(s, dtype=np.float64) for s in piv["serie"].to_list()]
    tn_tot = np.asarray(piv["tn_total"].to_list())
    series = [_serie(x) for x in series_raw]
    validos = [i for i, s in enumerate(series) if s is not None]
    rng = np.random.default_rng(PARAM["semilla"])

    # (a) elegir k sobre una muestra: silhouette (matriz DTW muestral) + balance
    m_k = min(PARAM["cl_muestra_k"], len(validos))
    idx_k = rng.choice(validos, size=m_k, replace=False)
    S = [series[i] for i in idx_k]
    D_S = _matriz(S)
    mejor = None
    for k in PARAM["cl_lista_k"]:
        lab_k, _ = _kmeans_dtw(S, k, PARAM["semilla"])
        tam = np.bincount(lab_k, minlength=k)
        balance = tam.min() / tam.sum()
        sil = silhouette_score(D_S, lab_k, metric="precomputed") if len(set(lab_k)) > 1 else -1
        ok = balance >= PARAM["cl_min_balance"]
        print(f"  k={k}: silhouette={sil:+.4f} balance={balance:.3f} {'ok' if ok else '(desbalanceado)'}")
        cand = (sil, k)
        if ok and (mejor is None or sil > mejor[0]):
            mejor = cand
    K = mejor[1] if mejor else PARAM["cl_lista_k"][0]
    print(f"  -> k elegido = {K}")

    # (b) ajustar centroides finales sobre cl_muestra_fit (o todas) y asignar TODAS
    pool = validos
    if PARAM["cl_muestra_fit"] and len(validos) > PARAM["cl_muestra_fit"]:
        orden = np.argsort(-tn_tot[validos])[:PARAM["cl_muestra_fit"]]
        pool = [validos[i] for i in orden]
    _, centros = _kmeans_dtw([series[i] for i in pool], K, PARAM["semilla"])

    lab_todos = np.zeros(len(ids), dtype=int)
    lab_todos[validos] = _asignar([series[i] for i in validos], centros)
    # series sin venta en <=corte -> cluster mayoritario
    if len(validos) < len(ids):
        mayor = int(np.bincount(lab_todos[validos]).argmax())
        for i in range(len(ids)):
            if series[i] is None:
                lab_todos[i] = mayor

    return pl.DataFrame({"agrupa_id": ids, "cluster_id": lab_todos.astype(np.int32)})


if PATH_CL.exists():
    print("[clusters] RESUME"); df_cl = pl.read_parquet(PATH_CL)
else:
    df_cl = _construir_clusters(df_fe)
    escribir_atomico(lambda t: df_cl.write_parquet(t), PATH_CL)
print("clusters:", df_cl.group_by("cluster_id").len().sort("cluster_id").to_dicts())

df_fe = df_fe.join(df_cl, on="agrupa_id", how="left").with_columns(
    pl.col("cluster_id").fill_null(df_cl["cluster_id"].min()).cast(pl.Int32))

## 6) Utilidades: WAPE a nivel producto + splits febreros

**NA de la clase**: periodo 201911 (t+2=202001) y 201912 (t+2=202002) tienen
`clase_raw` NA -> no entrenan; las de 201912 son las de inferencia. Entrenable =
`clase_raw` no nula (periodo <= 201910).

In [ ]:
def periodo_mas_h(p, h):
    y, m = divmod(p, 100); tot = y * 12 + (m - 1) + h
    return (tot // 12) * 100 + (tot % 12) + 1


def splits_febreros(periodos, h):
    pset = set(periodos)
    st = [p for p in periodos if periodo_mas_h(p, h) in pset and periodo_mas_h(p, h) % 100 == 2]
    return st or [sorted(periodos)[-1]]


def wape_producto(df_val: pd.DataFrame) -> float:
    d = df_val[df_val["product_id"].isin(PROD_TARGET)]
    if len(d) == 0:
        return float("nan")
    gg = d.groupby("product_id").agg(actual=("clase_raw", "sum"), fcst=("pred_tn", "sum"))
    den = gg["actual"].sum()
    return float("nan") if den <= 0 else float(np.abs(gg["actual"] - gg["fcst"]).sum() / den)

## 7) Optuna + entrenamiento final POR cluster

Un estudio Optuna por cluster (Tweedie, max_bin=1023, todos los núcleos),
optimizando el WAPE real en toneladas agregado a producto, validando por febreros.

In [ ]:
FEATS = PARAM["features"]
CATS = [c for c in PARAM["cols_categoricas"] if c in FEATS]


def _pd_cluster(k):
    d = df_fe.filter(pl.col("cluster_id") == k).to_pandas()
    for c in CATS:
        d[c] = d[c].astype("category")
    return d


def _params(trial, seed):
    p = {"objective": PARAM["objetivo_lgbm"], "metric": "tweedie", "max_bin": PARAM["max_bin"],
         "num_threads": PARAM["num_threads"], "verbosity": -1, "boosting_type": "gbdt", "seed": seed}
    if PARAM["objetivo_lgbm"] == "tweedie":
        lo, hi = PARAM["tweedie_vp_rango"]
        p["tweedie_variance_power"] = trial.suggest_float("tweedie_variance_power", lo, hi)
    p.update({
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 5e-3, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1500),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": 1,
    })
    return p


def _optuna_cluster(d, k):
    path = DIR_OUT / "optuna" / f"cluster_{k}.json"
    if path.exists():
        return leer_json(path)
    periodos = sorted(d["periodo"].unique().tolist())
    stands = [p for p in splits_febreros(periodos, PARAM["horizonte"]) if p < PARAM["periodo_inferencia"]][-2:]

    def objective(trial):
        pars = _params(trial, PARAM["semillas_ensemble"][0]); errs = []
        for st in stands:
            tr = d[(d["periodo"] < st) & d["clase_raw"].notna()]
            vl = d[(d["periodo"] == st) & d["clase_raw"].notna()].copy()
            if len(tr) == 0 or len(vl) == 0:
                continue
            m = lgb.LGBMRegressor(**pars)
            m.fit(tr[FEATS], tr["clase_scaled"], categorical_feature=CATS)
            vl["pred_tn"] = np.maximum(m.predict(vl[FEATS]), 0.0) * vl["s"].to_numpy()
            w = wape_producto(vl)
            if not np.isnan(w):
                errs.append(w)
        return float(np.mean(errs)) if errs else 1e9

    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=PARAM["semilla"]))
    study.optimize(objective, n_trials=PARAM["n_trials"], show_progress_bar=False)
    hiper = {"cluster": int(k), "wape_val": study.best_value, "params": study.best_params,
             "n_filas": int(len(d))}
    escribir_atomico(lambda t: Path(t).write_text(json.dumps(hiper, indent=2, default=str)), path)
    return hiper


def _entrenar_predecir(d, hiper):
    base = {"objective": PARAM["objetivo_lgbm"], "metric": "tweedie", "max_bin": PARAM["max_bin"],
            "num_threads": PARAM["num_threads"], "verbosity": -1, "boosting_type": "gbdt"}
    base.update(hiper["params"])
    tr = d[d["clase_raw"].notna()]
    inf = d[d["periodo"] == PARAM["periodo_inferencia"]].copy()
    if len(inf) == 0:
        return pd.DataFrame(columns=["product_id", "pred_tn"])
    preds = []
    for s in PARAM["semillas_ensemble"]:
        m = lgb.LGBMRegressor(**{**base, "seed": s})
        m.fit(tr[FEATS], tr["clase_scaled"], categorical_feature=CATS)
        preds.append(np.maximum(m.predict(inf[FEATS]), 0.0))
    inf["pred_tn"] = np.maximum(np.mean(preds, axis=0) * inf["s"].to_numpy(), PARAM["clip_min"])
    return inf[["product_id", "pred_tn"]]


PATH_PRED = DIR_OUT / "pred_base.parquet"
DIR_PREDK = DIR_OUT / "pred"

if PATH_PRED.exists():
    print("[pred_base] RESUME"); pred_base = pl.read_parquet(PATH_PRED).to_pandas()
else:
    # bucle por cluster con checkpoint POR cluster (spot-friendly): cada cluster deja
    # su optuna (optuna/cluster_k.json) y su prediccion (pred/cluster_k.parquet); una
    # preemption solo pierde el cluster en curso, no los ya terminados.
    for k in sorted(df_fe["cluster_id"].unique().to_list()):
        pth_k = DIR_PREDK / f"cluster_{k}.parquet"
        if pth_k.exists():
            print(f"  [cluster {k}] RESUME (pred ya existe)")
            continue
        d = _pd_cluster(k)
        if d["clase_raw"].notna().sum() == 0:
            escribir_atomico(lambda t: pl.DataFrame(
                {"product_id": pl.Series([], dtype=pl.Int64),
                 "pred_tn": pl.Series([], dtype=pl.Float64)}).write_parquet(t), pth_k)
            continue
        hip = _optuna_cluster(d, k)
        pr = _entrenar_predecir(d, hip)
        escribir_atomico(lambda t, pr=pr: pl.from_pandas(pr).write_parquet(t), pth_k)
        print(f"  cluster {k}: filas={hip['n_filas']:>8}  wape_val={hip['wape_val']:.4f}")

    partes = [pl.read_parquet(f).to_pandas() for f in sorted(DIR_PREDK.glob("cluster_*.parquet"))]
    partes = [p for p in partes if len(p)]
    pred_base = pd.concat(partes, ignore_index=True).groupby("product_id", as_index=False)["pred_tn"].sum()
    escribir_atomico(lambda t: pl.from_pandas(pred_base).write_parquet(t), PATH_PRED)

# resumen desde los checkpoints de optuna (robusto a resume)
resumen = [(h["cluster"], h["n_filas"], round(h["wape_val"], 4))
           for h in (leer_json(j) for j in sorted((DIR_OUT / "optuna").glob("cluster_*.json")))]
if resumen:
    print("resumen (cluster, filas, wape_val):", resumen)
    print("wape_val ponderado:", round(np.average([r[2] for r in resumen],
                                                  weights=[r[1] for r in resumen]), 4))

## 8) Submissions con magic multiplier (×0.9, ×1.0, ×1.1)

Se emiten 3 archivos para subir los tres a Kaggle y que el leaderboard elija.

In [ ]:
apr = pd.read_csv(DIR_RAW / "product_id_apredecir201912.txt", sep="\t")[["product_id"]]
base_sub = apr.merge(pred_base.rename(columns={"pred_tn": "tn"}), on="product_id", how="left")
base_sub["tn"] = base_sub["tn"].fillna(PARAM["clip_min"])

for mult in PARAM["multiplicadores"]:
    sub = base_sub.copy()
    sub["tn"] = np.maximum(sub["tn"] * mult, PARAM["clip_min"])
    ruta = DIR_OUT / f"submission_x{mult}.csv"
    escribir_atomico(lambda t, s=sub: s.to_csv(t, index=False), ruta)
    print(f"  submission ×{mult}: filas={len(sub)} tn_total={sub['tn'].sum():.1f} -> {ruta.name}")

if PARAM["submit"]:
    for mult in PARAM["multiplicadores"]:
        flag = DIR_OUT / "submits" / f"x{mult}.done"
        if flag.exists():
            print(f"  submit ×{mult} ya hecho -> se saltea"); continue
        ruta = DIR_OUT / f"submission_x{mult}.csv"
        try:
            r = subprocess.run(
                ["kaggle", "competitions", "submit", "-c", PARAM["kaggle_competition"],
                 "-f", str(ruta), "-m", f"pipe_v2 x{mult}"],
                capture_output=True, text=True)
            print(f"  submit ×{mult}: rc={r.returncode} {(r.stdout or r.stderr).strip()[:200]}")
            if r.returncode == 0:
                escribir_atomico(lambda t: Path(t).write_text(time.strftime("%Y-%m-%d %H:%M:%S")), flag)
        except Exception as e:
            print(f"  submit ×{mult} fallo (subir a mano el csv): {e}")

print("\nOK. Submissions en", DIR_OUT, "(submission_x*.csv)")